# 📘 Notebook 3: Class Attributes vs Instance Attributes

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. Distinguish between **class attributes** and **instance attributes**
2. Understand how Python's **attribute lookup chain** works
3. Use class attributes for **shared data** (e.g., constants, counters)
4. Track all instances using a **class-level list**
5. Inspect attributes using `__dict__`

---

## 1. Two Types of Attributes

Python classes have two kinds of attributes:

| Type | Defined Where | Shared? | Example Use Case |
|------|--------------|---------|------------------|
| **Class Attribute** | Inside the class body, outside any method | ✅ Shared by ALL instances | Discount rate, counter, constants |
| **Instance Attribute** | Inside `__init__` (via `self.x = ...`) | ❌ Unique to each instance | Name, price, quantity |

```python
class Item:
    pay_rate = 0.8          # ← CLASS attribute (shared by all)
    
    def __init__(self, name, price):
        self.name = name    # ← INSTANCE attribute (unique to each)
        self.price = price  # ← INSTANCE attribute (unique to each)
```

In [ ]:
class Item:
    # Class attribute — shared by ALL instances
    pay_rate = 0.8  # 20% discount (pay 80% of original price)
    
    def __init__(self, name: str, price: float, quantity: int = 0):
        # Instance attributes — unique to EACH instance
        self.name = name
        self.price = price
        self.quantity = quantity

item1 = Item("Phone", 100, 5)
item2 = Item("Laptop", 1000, 3)

# Both instances share the same pay_rate
print(f"item1.pay_rate = {item1.pay_rate}")
print(f"item2.pay_rate = {item2.pay_rate}")
print(f"Item.pay_rate  = {Item.pay_rate}")
print(f"\nAll the same? {item1.pay_rate == item2.pay_rate == Item.pay_rate}")

---

## 2. The Attribute Lookup Chain 🔍

When you access `item1.pay_rate`, Python searches in this order:

```
1. Instance's __dict__    →  Found? Return it.
         ↓ Not found
2. Class's __dict__       →  Found? Return it.
         ↓ Not found
3. Parent class's __dict__ → Found? Return it.
         ↓ Not found
4. AttributeError!
```

Let's verify this with `__dict__`:

In [ ]:
class Item:
    pay_rate = 0.8
    
    def __init__(self, name: str, price: float):
        self.name = name
        self.price = price

item1 = Item("Phone", 100)

# Instance's own attributes
print("Instance __dict__:")
print(f"  {item1.__dict__}")
# Notice: pay_rate is NOT in the instance dict!

print("\nClass __dict__ (relevant parts):")
class_attrs = {k: v for k, v in Item.__dict__.items() if not k.startswith('_')}
print(f"  {class_attrs}")
# pay_rate IS in the class dict!

print(f"\nitem1.pay_rate = {item1.pay_rate}")
print("↑ Found via the class, not the instance!")

### What happens when you assign to an instance?

Assigning `item1.pay_rate = 0.7` creates a **new instance attribute** that **shadows** the class attribute:

In [ ]:
class Item:
    pay_rate = 0.8
    
    def __init__(self, name: str, price: float):
        self.name = name
        self.price = price

item1 = Item("Phone", 100)
item2 = Item("Laptop", 1000)

print("=== Before modification ===")
print(f"item1.pay_rate = {item1.pay_rate}  (from class)")
print(f"item2.pay_rate = {item2.pay_rate}  (from class)")

# Override for item1 ONLY
item1.pay_rate = 0.7  # Creates an INSTANCE attribute on item1

print("\n=== After item1.pay_rate = 0.7 ===")
print(f"item1.pay_rate = {item1.pay_rate}  (from instance — shadowed!)")
print(f"item2.pay_rate = {item2.pay_rate}  (still from class — unchanged)")
print(f"Item.pay_rate  = {Item.pay_rate}   (class attribute — unchanged)")

print("\n=== Proof via __dict__ ===")
print(f"item1.__dict__ = {item1.__dict__}")  # Has pay_rate!
print(f"item2.__dict__ = {item2.__dict__}")  # Does NOT have pay_rate

---

## 3. Practical Example: Applying Discounts

Let's use `pay_rate` as a class attribute to apply discounts.

### `self.pay_rate` vs `Item.pay_rate`

Inside a method, you have a choice:

| Access Pattern | Behavior |
|---------------|----------|
| `self.pay_rate` | Checks instance first, then class (allows per-instance overrides) |
| `Item.pay_rate` | Always uses the class-level value (ignores instance overrides) |

In [ ]:
class Item:
    pay_rate = 0.8  # All items get 20% off by default
    
    def __init__(self, name: str, price: float, quantity: int = 0):
        self.name = name
        self.price = price
        self.quantity = quantity
    
    def apply_discount(self):
        """Apply discount using self.pay_rate (allows per-instance overrides)."""
        self.price = self.price * self.pay_rate  # ← Uses self, not Item

item1 = Item("Phone", 100)
item2 = Item("Laptop", 1000)

# Give the laptop a special 30% discount
item2.pay_rate = 0.7

# Apply discounts
item1.apply_discount()
item2.apply_discount()

print(f"{item1.name}: ${item1.price} (20% off — used class pay_rate)")
print(f"{item2.name}: ${item2.price} (30% off — used instance pay_rate)")

---

## 4. Tracking All Instances with a Class Attribute

A powerful pattern is using a **class-level list** to keep track of all instances ever created:

In [ ]:
class Item:
    pay_rate = 0.8
    all = []  # Class attribute: list of ALL instances
    
    def __init__(self, name: str, price: float, quantity: int = 0):
        assert price >= 0, f"Price {price} is not >= 0!"
        assert quantity >= 0, f"Quantity {quantity} is not >= 0!"
        
        self.name = name
        self.price = price
        self.quantity = quantity
        
        # Register this instance in the class-level list
        Item.all.append(self)
    
    def calculate_total_price(self):
        return self.price * self.quantity
    
    def __repr__(self):
        return f"Item('{self.name}', {self.price}, {self.quantity})"

# Create several items
item1 = Item("Phone", 100, 5)
item2 = Item("Laptop", 1000, 3)
item3 = Item("Cable", 10, 50)
item4 = Item("Mouse", 50, 10)
item5 = Item("Keyboard", 75, 8)

# Access ALL instances through the class
print("All items:")
for item in Item.all:
    print(f"  {item}")

print(f"\nTotal items created: {len(Item.all)}")

# Find the most expensive item
most_expensive = max(Item.all, key=lambda x: x.price)
print(f"Most expensive: {most_expensive.name} (${most_expensive.price})")

---

## 5. Common Class Attribute Patterns

### Pattern 1: Constants

In [ ]:
class Circle:
    PI = 3.14159265358979  # Constant shared by all circles
    
    def __init__(self, radius: float):
        self.radius = radius
    
    def area(self):
        return Circle.PI * self.radius ** 2  # Using class constant

c1 = Circle(5)
c2 = Circle(10)
print(f"Circle 1 area: {c1.area():.2f}")
print(f"Circle 2 area: {c2.area():.2f}")

### Pattern 2: Instance Counter

In [ ]:
class User:
    count = 0  # Tracks how many users have been created
    
    def __init__(self, username: str):
        self.username = username
        User.count += 1  # Increment the CLASS attribute, not instance!
        self.user_id = User.count  # Assign a unique ID
    
    def __repr__(self):
        return f"User(id={self.user_id}, username='{self.username}')"

u1 = User("alice")
u2 = User("bob")
u3 = User("charlie")

print(u1)
print(u2)
print(u3)
print(f"\nTotal users created: {User.count}")

### Pattern 3: Configuration Defaults

In [ ]:
class DatabaseConnection:
    # Default configuration (class attributes)
    host = "localhost"
    port = 5432
    timeout = 30
    
    def __init__(self, database: str, username: str):
        self.database = database
        self.username = username
    
    def connection_string(self):
        return f"postgresql://{self.username}@{self.host}:{self.port}/{self.database}"

# Default connection
db1 = DatabaseConnection("mydb", "admin")
print(f"Default: {db1.connection_string()}")

# Custom host for a specific connection
db2 = DatabaseConnection("prod_db", "admin")
db2.host = "192.168.1.100"  # Override for this instance only
db2.port = 5433
print(f"Custom:  {db2.connection_string()}")

# db1 still uses defaults
print(f"Still default: {db1.connection_string()}")

---

## 6. ⚠️ Mutable Class Attribute Gotcha

Be careful with **mutable** class attributes (lists, dicts). They're shared by reference!

In [ ]:
# DANGEROUS: Mutable class attribute
class StudentBad:
    grades = []  # Shared mutable list — this is a bug!
    
    def __init__(self, name):
        self.name = name
    
    def add_grade(self, grade):
        self.grades.append(grade)  # Modifying the SHARED list!

s1 = StudentBad("Alice")
s2 = StudentBad("Bob")

s1.add_grade(95)
s2.add_grade(87)

print(f"Alice's grades: {s1.grades}")  # [95, 87] ← Bug! Alice has Bob's grade!
print(f"Bob's grades:   {s2.grades}")  # [95, 87] ← Same list!
print(f"Same object? {s1.grades is s2.grades}")  # True — they share the same list!

In [ ]:
# CORRECT: Mutable attributes should be instance attributes
class StudentGood:
    def __init__(self, name):
        self.name = name
        self.grades = []  # Each instance gets its OWN list
    
    def add_grade(self, grade):
        self.grades.append(grade)

s1 = StudentGood("Alice")
s2 = StudentGood("Bob")

s1.add_grade(95)
s2.add_grade(87)

print(f"Alice's grades: {s1.grades}")  # [95] ✅
print(f"Bob's grades:   {s2.grades}")  # [87] ✅
print(f"Same object? {s1.grades is s2.grades}")  # False — different lists!

> 🔑 **Rule of Thumb**: Use class attributes for **immutable** shared data (numbers, strings, tuples). Use instance attributes for **mutable** per-object data (lists, dicts, sets). The exception is intentionally shared collections like `Item.all` above.

---

## 🏋️ Practice Exercises

### Exercise 1: Product Inventory
Create a `Product` class with:
- A class attribute `tax_rate = 0.08` (8% tax)
- A class attribute `all_products = []`
- Instance attributes: `name`, `price`, `quantity`
- Method `price_after_tax()` that returns price with tax
- Track all products in `all_products`

In [ ]:
# Exercise 1: Your code here



### Exercise 2: Class Attribute vs Instance Attribute
Predict the output of this code, then run it to verify:

In [ ]:
# Exercise 2: Predict the output BEFORE running!
class Config:
    debug = False
    version = "1.0"

c1 = Config()
c2 = Config()

c1.debug = True
Config.version = "2.0"

print(f"c1.debug = {c1.debug}")       # Your prediction: ?
print(f"c2.debug = {c2.debug}")       # Your prediction: ?
print(f"c1.version = {c1.version}")   # Your prediction: ?
print(f"c2.version = {c2.version}")   # Your prediction: ?

---

### ✅ Solutions

In [ ]:
# Solution 1
class Product:
    tax_rate = 0.08
    all_products = []
    
    def __init__(self, name: str, price: float, quantity: int = 0):
        self.name = name
        self.price = price
        self.quantity = quantity
        Product.all_products.append(self)
    
    def price_after_tax(self):
        return self.price * (1 + self.tax_rate)
    
    def __repr__(self):
        return f"Product('{self.name}', ${self.price})"

p1 = Product("Widget", 10.00, 100)
p2 = Product("Gadget", 25.00, 50)

for p in Product.all_products:
    print(f"{p.name}: ${p.price} → ${p.price_after_tax():.2f} (with tax)")

In [ ]:
# Solution 2 Explanation:
# c1.debug = True      → Creates an INSTANCE attribute on c1 (shadows class attr)
# c2.debug              → Still reads from CLASS (False)
# Config.version = "2.0" → Changes the CLASS attribute
# c1.version            → No instance attr, reads from class → "2.0"
# c2.version            → No instance attr, reads from class → "2.0"

print("Expected output:")
print("c1.debug = True")
print("c2.debug = False")
print('c1.version = 2.0')
print('c2.version = 2.0')

---

## 📌 Key Takeaways

1. **Class attributes** are shared by all instances; **instance attributes** are unique per object
2. Python searches **instance → class → parent class** when looking up attributes
3. Assigning to `self.x` creates/overrides an **instance** attribute (shadowing)
4. Use `Item.x` to access/modify the **class** attribute directly
5. **Mutable class attributes** (lists, dicts) are shared by reference — usually a bug!
6. `__dict__` lets you inspect what attributes exist at each level

---

**⏭️ Next: [Notebook 04 — Class Methods & Static Methods](./04_Class_and_Static_Methods.ipynb)** — Learn about `@classmethod`, `@staticmethod`, and when to use each!